# PatchTST explainability on synthetic forecasting data

This notebook trains a compact Hugging Face PatchTST from random initialization, then demonstrates XPC's synthetic-data, parameter-inspection, error, PDP, first-order ALE, grouped Shapley, and waterfall helpers.

## 1. Environment and project paths

In [ ]:
from pathlib import Path
import sys

on_drive = False

if on_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    project_root = Path("/content/drive/MyDrive/Recherche/Thèse Gaspard/Codes/xtsf")
    data_path = Path("/content/drive/MyDrive/Recherche/Thèse Gaspard/Datasets")
else:
    project_root = Path.cwd().resolve()
    if project_root.name == "src":
        project_root = project_root.parent
    data_path = project_root / "datasets"

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
%cd $project_root

## 2. Imports and one run seed

Install the notebook dependencies with `pip install -e ".[notebook]"` in a user-prepared environment. No checkpoint or external dataset is downloaded.

In [ ]:
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import PatchTSTConfig, PatchTSTForPrediction

from xpc import (
    BaselineMasker,
    DataSpec,
    FeatureGroups,
    ShapleyExplainer,
    TorchModelAdapter,
    accumulated_local_effects,
    error_summary,
    make_synthetic_forecasting_data,
    parameter_counts,
    parameter_structure,
    partial_dependence,
    plot_accumulated_local_effects,
    plot_partial_dependence,
    plot_prediction_errors,
    plot_shapley_waterfall,
)

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_path = project_root / "outputs"
output_path.mkdir(exist_ok=True)
device

## 3. Package-provided synthetic forecasting data

`make_synthetic_forecasting_data` returns the complete series, named structural components, and chronological `(sample, context, channel)` / `(sample, horizon, channel)` windows.

In [ ]:
CONTEXT_LENGTH = 48
HORIZON = 6
synthetic = make_synthetic_forecasting_data(
    n_steps=24 * 28,
    context_length=CONTEXT_LENGTH,
    horizon=HORIZON,
    seed=SEED,
)

figure, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(synthetic.time, synthetic.values, color="black", linewidth=1)
axes[0].set_ylabel("Synthetic target")
for name in ("trend", "daily", "weekly", "interaction"):
    axes[1].plot(synthetic.time, synthetic.components[name], label=name)
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Component")
axes[1].legend(ncol=4)
figure.suptitle("Synthetic trend and seasonality")
figure.tight_layout()
plt.show()

split = int(0.75 * len(synthetic.contexts))
train_contexts, test_contexts = synthetic.contexts[:split], synthetic.contexts[split:]
train_targets, test_targets = synthetic.targets[:split], synthetic.targets[split:]
train_contexts.shape, train_targets.shape, test_contexts.shape

## 4. Compact PatchTST training

This is a small pedagogical configuration, not a benchmark configuration. The chronological test tail is never used for fitting.

In [ ]:
config = PatchTSTConfig(
    num_input_channels=1,
    context_length=CONTEXT_LENGTH,
    prediction_length=HORIZON,
    patch_length=8,
    patch_stride=4,
    num_hidden_layers=1,
    d_model=32,
    num_attention_heads=4,
    ffn_dim=64,
    attention_dropout=0.0,
    ff_dropout=0.0,
    head_dropout=0.0,
    scaling="std",
    loss="mse",
)
patchtst = PatchTSTForPrediction(config).to(device)
optimizer = torch.optim.AdamW(patchtst.parameters(), lr=2e-3)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    TensorDataset(
        torch.as_tensor(train_contexts, dtype=torch.float32),
        torch.as_tensor(train_targets, dtype=torch.float32),
    ),
    batch_size=64,
    shuffle=True,
    generator=loader_generator,
)

loss_history = []
for epoch in range(5):
    patchtst.train()
    epoch_losses = []
    for past_values, future_values in train_loader:
        past_values = past_values.to(device)
        future_values = future_values.to(device)
        optimizer.zero_grad()
        output = patchtst(past_values=past_values, future_values=future_values)
        output.loss.backward()
        optimizer.step()
        epoch_losses.append(float(output.loss.detach().cpu()))
    loss_history.append(float(np.mean(epoch_losses)))
    print(f"epoch {epoch + 1}: loss={loss_history[-1]:.5f}")
patchtst.eval()

## 5. Parameter counts and structure

The inspection helpers use `named_parameters()` and report metadata without copying the actual weight values.

In [ ]:
counts = parameter_counts(patchtst)
structure = pd.DataFrame(asdict(item) for item in parameter_structure(patchtst))
print(counts)
structure.head(12)

## 6. Flat-window adapter and forecast errors

XPC's tabular boundary treats every lag as a feature and every forecast step as an output. The wrapper restores PatchTST's three-dimensional input internally.

In [ ]:
class FlatPatchTST(torch.nn.Module):
    def __init__(self, backbone, context_length):
        super().__init__()
        self.backbone = backbone
        self.context_length = context_length

    def forward(self, values):
        past_values = values.reshape(-1, self.context_length, 1)
        return self.backbone(past_values=past_values).prediction_outputs[..., 0]


flat_model = FlatPatchTST(patchtst, CONTEXT_LENGTH).to(device).eval()
adapter = TorchModelAdapter(flat_model, device=str(device))
train_flat = train_contexts[..., 0]
test_flat = test_contexts[..., 0]
test_truth = test_targets[..., 0]
test_predictions = adapter.predict(test_flat)
error_summary(test_truth, test_predictions)

In [ ]:
error_figure, _ = plot_prediction_errors(
    test_truth,
    test_predictions,
    output=0,
    title="PatchTST first-horizon errors",
)
plt.show()

## 7. Partial dependence and first-order ALE

The selected feature is the most recent lag. PDP replaces it globally; ALE measures local prediction differences inside empirical quantile bins. Both curves are associational diagnostics, not causal effects.

In [ ]:
lag_names = [f"lag_{offset}" for offset in range(-CONTEXT_LENGTH + 1, 1)]
effect_background = train_flat[-128:]
pdp = partial_dependence(
    adapter,
    effect_background,
    "lag_0",
    feature_names=lag_names,
    n_points=15,
)
ale = accumulated_local_effects(
    adapter,
    effect_background,
    "lag_0",
    feature_names=lag_names,
    n_bins=8,
)

pdp_figure, _ = plot_partial_dependence(pdp, output=0)
ale_figure, _ = plot_accumulated_local_effects(ale, output=0)
plt.show()

## 8. Grouped Shapley waterfall

The 48 lags become three temporal players. The waterfall uses raw signed Shapley values; any finite-sampling reconstruction residual is shown separately from the model prediction.

In [ ]:
lag_groups = FeatureGroups(
    {
        "oldest 16 lags": range(0, 16),
        "middle 16 lags": range(16, 32),
        "recent 16 lags": range(32, 48),
    },
    remaining="ignore",
)
explanation = ShapleyExplainer(
    adapter,
    BaselineMasker("mean", background=train_flat),
    data_spec=DataSpec(feature_names=lag_names),
    feature_groups=lag_groups,
    n_coalitions=8,
    random_state=SEED,
    heighten=False,
)(test_flat[:1])

waterfall_figure, _ = plot_shapley_waterfall(
    explanation, unit=0, output=0, max_display=None
)
print("efficiency residual:", explanation.efficiency_residual[0, 0])
plt.show()

## 9. Reading the diagnostics

- Parameter structure describes the implementation, not feature influence.
- Error plots evaluate predictive behavior on the chronological test tail.
- PDP and ALE summarize the response to one lag over a background population. Correlated lags can make PDP combinations unrealistic; ALE reduces, but does not eliminate, interpretation risks.
- The Shapley waterfall explains one forecast step for one context under the declared baseline and temporal grouping. It is not a statement about attention weights or causality.